# BƯỚC 1 & 2: KHẢO SÁT & KIỂM TRA CHẤT LƯỢNG DỮ LIỆU THÔ (DATA QUALITY & PROFILING)

---
**Mục tiêu của Notebook:**
1. **Trích xuất dữ liệu cần thiết (Feature Selection)** từ kho dữ liệu thô Bronze (bronze_listings + bronze_details_valid).
2. **Kiểm tra cấu trúc và kiểu dữ liệu (Schema & Data Types)** của toàn bộ các trường tuyển dụng.
3. **Đo lường giá trị khuyết (Missing / Null Report)**: Thống kê số lượng Null/None và chuỗi rỗng trên từng cột.
4. **Kiểm tra trùng lặp (Duplicates Report)**: Phát hiện trùng lặp theo ID nguồn, URL bài đăng và thực thể công việc [Công ty + Chức danh].
5. Đưa ra cơ sở thực tế để thực hiện bước Tiền xử lý & Làm sạch dữ liệu tiếp theo.

In [1]:
import os
import json
import pandas as pd
import numpy as np

from IPython.display import display

# Cấu hình hiển thị pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
print(' Môi trường phân tích dữ liệu đã sẵn sàng!')

 Môi trường phân tích dữ liệu đã sẵn sàng!


## 1. Nạp Dữ Liệu Master Từ Tầng Bronze

Đọc 2 tệp tích lũy chuẩn từ `data/bronze/`:
- `bronze_listings_master.json`: Bảng danh mục tin tuyển dụng (kèm tags, logo, trạng thái crawl).
- `bronze_details_valid.json`: Bảng chi tiết việc làm hợp lệ sau khi qua bộ kiểm định Data Quality.

In [2]:
# Duong dan toi 2 file Bronze Master (tuong thich moi thu muc lam viec)
def resolve_path(rel_subpath):
    candidates = [
        rel_subpath,
        f"../{rel_subpath}",
        f"../../{rel_subpath}"
    ]
    for c in candidates:
        if os.path.exists(c):
            return os.path.abspath(c)
    return os.path.abspath(rel_subpath)

listings_path = resolve_path('data/bronze/bronze_listings_master.json')
details_path = resolve_path('data/bronze/bronze_details_valid.json')

with open(listings_path, 'r', encoding='utf-8') as f:
    listings_data = json.load(f)
with open(details_path, 'r', encoding='utf-8') as f:
    details_data = json.load(f)

df_listings = pd.DataFrame(listings_data)
df_details = pd.DataFrame(details_data)

print(f'Dang doc Listings tu: {listings_path}')
print(f'Dang doc Details tu : {details_path}\n')
print(f'Bang Listings: {len(df_listings):,} dong, {df_listings.shape[1]} cot')
print(f'Bang Details : {len(df_details):,} dong, {df_details.shape[1]} cot\n')

# Thong ke so luong tin theo tung san
summary_sources = pd.DataFrame({
    'So Listings': df_listings['source'].value_counts(),
    'So Details Valid': df_details['source'].value_counts()
})
display(summary_sources)


Dang doc Listings tu: C:\Users\Minh Doan\repo_github\IT-Job-Market-Intelligence\data\bronze\bronze_listings_master.json
Dang doc Details tu : C:\Users\Minh Doan\repo_github\IT-Job-Market-Intelligence\data\bronze\bronze_details_valid.json

Bang Listings: 5,666 dong, 20 cot
Bang Details : 5,631 dong, 35 cot



,So Listings,So Details Valid
source,,
topcv,3695,3661
vietnamworks,1786,1785
topdev,185,185


## 2. Ghép 2 Bảng (Join Listings + Details)

Ghép theo cặp khóa định danh duy nhất: `['source', 'source_job_id']`.
- **Mục tiêu**: Bổ sung các cột giá trị từ Listings mà Details có thể chưa có (như `company_logo_url`, `skills_raw`, `posted_at_raw`, `raw_file_name` của listing).
- Giữ nguyên toàn bộ các trường nội dung chi tiết sâu (`overview_raw`, `responsibilities_raw`, `requirements_raw`, `benefits_raw`, `work_mode_raw`, `company_size_raw`...) từ Details.

In [3]:
# Chọn các cột bổ sung từ Listings (tránh xung đột trùng lặp cột)
listing_extra_cols = ['source', 'source_job_id', 'company_logo_url', 'skills_raw', 'posted_at_raw', 'raw_file_name']
df_list_sub = df_listings[listing_extra_cols].drop_duplicates(subset=['source', 'source_job_id']).rename(
    columns={'raw_file_name': 'listing_file_name'}
)

# Ghép bảng Details với Listings theo ['source', 'source_job_id']
df_merged = pd.merge(
    df_details,
    df_list_sub,
    on=['source', 'source_job_id'],
    how='inner'
)

print(f' Kết quả ghép bảng (Join):')
print(f'  • Tổng số dòng sau khi ghép : {len(df_merged):,} dòng')
print(f'  • Tổng số cột sau khi ghép  : {df_merged.shape[1]} cột')
match_rate = len(df_merged) / len(df_details) * 100
print(f'  • Tỷ lệ khớp khóa (Match)  : {match_rate:.2f}% ({len(df_merged):,}/{len(df_details):,} tin chi tiết có listing tương ứng)')

# Ghép kiểu 'inner' sẽ BỎ các tin chi tiết không có listing ➔ liệt kê ra thay vì bỏ qua trong im lặng
key = ['source', 'source_job_id']
unmatched = df_details.merge(df_list_sub[key], on=key, how='left', indicator=True).query('_merge == "left_only"')
if len(unmatched):
    print(f'  ⚠ {len(unmatched):,} tin chi tiết KHÔNG có listing (bị loại khỏi bảng ghép), theo nguồn:',
          unmatched['source'].value_counts().to_dict())
    display(unmatched[key + ['title_raw', 'company_name_raw', 'crawled_at']].head(10))
else:
    print('  • Mọi tin chi tiết đều có listing tương ứng.')

 Kết quả ghép bảng (Join):
  • Tổng số dòng sau khi ghép : 5,631 dòng
  • Tổng số cột sau khi ghép  : 39 cột
  • Tỷ lệ khớp khóa (Match)  : 100.00% (5,631/5,631 tin chi tiết có listing tương ứng)
  • Mọi tin chi tiết đều có listing tương ứng.


## 3. Thống Kê Chi Tiết: Info, Missing Values (Null / Rỗng), và Duplicates (Dup)

Khảo sát toàn diện chất lượng dữ liệu của bảng sau khi ghép:
1. **Cấu trúc & Kiểu dữ liệu (`.info()`)**.
2. **Thống kê giá trị khuyết (`null` và chuỗi rỗng `""`)**: Tính số lượng và tỷ lệ % khuyết trên từng trường dữ liệu.
3. **Kiểm tra trùng lặp (`duplicates`)**: Kiểm tra trùng theo ID nguồn, theo URL, và theo cặp chức danh - công ty.

In [4]:
# 1. Hiển thị thông tin tổng quan kiểu dữ liệu
print('--- 1. THÔNG TIN CẤU TRÚC (DF INFO) ---')
df_merged.info(verbose=True, show_counts=True)

# 2. Thống kê Missing Values (Null / None và Chuỗi Rỗng "")
null_counts = df_merged.isnull().sum()
empty_counts = (df_merged == '').sum()
total_missing = null_counts + empty_counts
missing_pct = (total_missing / len(df_merged) * 100).round(2)

df_missing_report = pd.DataFrame({
    'Kiểu Dữ Liệu': df_merged.dtypes.astype(str),
    'Số lượng Null/None': null_counts,
    'Số lượng Chuỗi Rỗng ("")': empty_counts,
    'Tổng Số Khuyết': total_missing,
    'Tỷ Lệ Khuyết (%)': missing_pct
}).sort_values(by='Tỷ Lệ Khuyết (%)', ascending=False)

print('\n--- 2. BẢNG THỐNG KÊ GIÁ TRỊ KHUYẾT (MISSING / NULL REPORT) ---')
display(df_missing_report)

# 3. Kiểm tra Trùng lặp (Duplicates Report)
dup_source_id = df_merged.duplicated(subset=['source', 'source_job_id']).sum()
dup_job_url = df_merged.duplicated(subset=['job_url']).sum()
dup_title_company = df_merged.duplicated(subset=['title_raw', 'company_name_raw']).sum()

print('--- 3. BẢNG KIỂM TRA TRÙNG LẶP (DUPLICATES REPORT) ---')
df_dup_report = pd.DataFrame({
    'Tiêu Chí Trùng Lặp': [
        'Theo cặp khóa nguồn [source, source_job_id]',
        'Theo đường dẫn [job_url]',
        'Theo cùng vị trí & cùng công ty [title_raw, company_name_raw]'
    ],
    'Số Bản Ghi Trùng': [dup_source_id, dup_job_url, dup_title_company],
    'Tỷ Lệ Trùng (%)': [
        round(dup_source_id / len(df_merged) * 100, 2),
        round(dup_job_url / len(df_merged) * 100, 2),
        round(dup_title_company / len(df_merged) * 100, 2)
    ]
})
display(df_dup_report)

# 4. Bản ghi trong Bronze KHÔNG có dấu vết kiểm định (dq_status trống): tức là được đưa vào Bronze mà không qua QualityValidator
no_dq = df_merged[df_merged['dq_status'].isna()] if 'dq_status' in df_merged else df_merged.iloc[:0]
print(f'--- 4. BẢN GHI THIẾU THÔNG TIN KIỂM ĐỊNH (dq_status trống): {len(no_dq):,} ---')
if len(no_dq):
    display(no_dq[['source', 'source_job_id', 'title_raw', 'company_name_raw', 'location_raw', 'crawled_at']])

--- 1. THÔNG TIN CẤU TRÚC (DF INFO) ---
<class 'pandas.DataFrame'>
RangeIndex: 5631 entries, 0 to 5630
Data columns (total 39 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   detail_id             5631 non-null   str   
 1   source                5631 non-null   str   
 2   source_job_id         5631 non-null   str   
 3   job_url               5631 non-null   str   
 4   title_raw             5631 non-null   str   
 5   company_name_raw      5631 non-null   str   
 6   company_url           5631 non-null   str   
 7   company_size_raw      5631 non-null   str   
 8   company_location_raw  5631 non-null   str   
 9   industry_raw          5631 non-null   str   
 10  salary_raw            5631 non-null   str   
 11  experience_raw        5631 non-null   str   
 12  education_raw         5631 non-null   str   
 13  location_raw          5631 non-null   str   
 14  work_mode_raw         5631 non-null   str   
 15  employmen

,Kiểu Dữ Liệu,Số lượng Null/None,"Số lượng Chuỗi Rỗng ("""")",Tổng Số Khuyết,Tỷ Lệ Khuyết (%)
preferred_skills_raw,str,0,5631,5631,100.00
updated_at_raw,str,0,3846,3846,68.30
overview_raw,str,0,3515,3515,62.42
company_size_raw,str,0,1161,1161,20.62
education_raw,str,0,867,867,15.40
working_time_raw,str,0,330,330,5.86
work_mode_raw,str,0,50,50,0.89
company_logo_url,str,0,39,39,0.69
requirements_raw,str,0,27,27,0.48
required_skills_raw,str,0,5,5,0.09


--- 3. BẢNG KIỂM TRA TRÙNG LẶP (DUPLICATES REPORT) ---


,Tiêu Chí Trùng Lặp,Số Bản Ghi Trùng,Tỷ Lệ Trùng (%)
0,"Theo cặp khóa nguồn [source, source_job_id]",0,0.00
1,Theo đường dẫn [job_url],0,0.00
2,"Theo cùng vị trí & cùng công ty [title_raw, company_name...",309,5.49


--- 4. BẢN GHI THIẾU THÔNG TIN KIỂM ĐỊNH (dq_status trống): 0 ---


## 4. Xem Mẫu Dữ Liệu Sau Khi Ghép (Toàn Bộ Các Cột - Dễ Quan Sát)

Vì dữ liệu sau khi ghép có **khoảng 40 cột** (số chính xác in ở mục 2; bao gồm cả các trường text dài và metadata), nếu để một bảng phẳng nằm ngang sẽ rất khó quan sát. Chúng ta tổ chức hiển thị theo các cách sau để bạn dễ kiểm tra nhất:

1.  **Cách 1: Hiển thị Bảng Chuyển Vị (Transposed `.T`)**: Chuyển toàn bộ các cột thành danh sách dọc. Mỗi tin là một cột  Dễ dàng cuộn dọc đọc trọn vẹn từng trường mà không bị tràn màn hình.
2.  **Cách 2: Hiển thị 10 Dòng Theo Từng Nhóm Nghiệp Vụ**: Phân tách các cột thành 5 nhóm có cấu trúc (*Định danh, Công ty, Lương & Đãi ngộ, Kỹ năng, Nội dung JD, Data Quality*).
3.  **Cách 3: Bảng Phẳng Đầy Đủ (toàn bộ các cột x 10 dòng)**: Mở rộng `display.max_columns = None` để bạn có thể cuộn ngang xem tất cả các cột nếu cần.

In [5]:
# --- CÁCH 1: HIỂN THỊ DẠNG CHUYỂN VỊ (TRANSPOSED VIEW) ---
# Các cột sẽ hiển thị theo chiều dọc (Index), các mẫu tin hiển thị theo cột
print(f' [CÁCH 1] XEM DỌC TOÀN BỘ {df_merged.shape[1]} CỘT CỦA 3 TIN TUYỂN DỤNG MẪU:')
sample_3 = df_merged.sample(3, random_state=42)
display(sample_3.T)


 [CÁCH 1] XEM DỌC TOÀN BỘ 39 CỘT CỦA 3 TIN TUYỂN DỤNG MẪU:


,3135,1835,3832
detail_id,topcv_2000065,vietnamworks_2118281,topcv_2046627
source,topcv,vietnamworks,topcv
source_job_id,2000065,2118281,2046627
job_url,https://www.topcv.vn/viec-lam/nhan-vien-kinh-doanh-phan-...,https://www.vietnamworks.com/ky-su-backend-2118281-jv,https://www.topcv.vn/viec-lam/truong-phong-lap-trinh/204...
title_raw,Nhân Viên Kinh Doanh Phần Mềm (Website),Kỹ Sư Backend,Trưởng Phòng Lập Trình
company_name_raw,CÔNG TY TNHH THƯƠNG MẠI DỊCH VỤ WEB IDEAS,Điện Toán Group,CÔNG TY CỔ PHẦN LIFETEX
company_url,https://www.topcv.vn/cong-ty/cong-ty-tnhh-thuong-mai-dic...,https://www.vietnamworks.com/nha-tuyen-dung/đien-toan-gr...,https://www.topcv.vn/cong-ty/cong-ty-co-phan-lifetex/178...
company_size_raw,10-24 nhân viên,,25-99 nhân viên
company_location_raw,"361 Đường Phạm Văn Bạch, Phường 15, Quận Tân Bình, Thành...","437/16 Hoàng Văn Thụ, Phường Tân Sơn Nhất, Thành phố Hồ ...","Số 15 Tố Hữu, Thanh Xuân, Hà Nội"
industry_raw,"Kinh doanh / Bán hàng, IT Phần cứng / Mạng, Công nghệ th...",Phần Mềm CNTT/Dịch vụ Phần mềm (Công Nghệ Thông Tin/Viễn...,"IT phần mềm, Công nghệ thông tin"


In [6]:
# ---  XEM MẪU 10 DÒNG THEO TỪNG NHÓM NGHIỆP VỤ ---
sample_10 = df_merged.sample(10, random_state=42)

# Nhóm 1: Định danh & Truy vết nguồn (7 cột)
group_identity = ['source', 'source_job_id', 'job_url', 'crawled_at', 'raw_date', 'raw_file_name', 'listing_file_name']
print(' NHÓM 1: ĐỊNH DANH & TRUY VẾT NGUỒN (7 CỘT)')
display(sample_10[group_identity])

# Nhóm 2: Thông tin Công ty (7 cột)
group_company = ['company_name_raw', 'company_url', 'company_logo_url', 'company_size_raw', 'company_location_raw', 'industry_raw', 'location_raw']
print('\n NHÓM 2: THÔNG TIN DOANH NGHIỆP (7 CỘT)')
display(sample_10[group_company])

# Nhóm 3: Vị trí, Lương & Đãi ngộ tuyển dụng (8 cột)
group_job_specs = ['title_raw', 'salary_raw', 'experience_raw', 'education_raw', 'work_mode_raw', 'employment_type_raw', 'working_time_raw', 'deadline_raw']
print('\n NHÓM 3: VỊ TRÍ, LƯƠNG & ĐIỀU KIỆN TUYỂN DỤNG (8 CỘT)')
display(sample_10[group_job_specs])

# Nhóm 4: Kỹ năng & Ngày tháng (6 cột)
group_skills = ['skills_raw', 'required_skills_raw', 'preferred_skills_raw', 'posted_at_raw', 'published_at_raw', 'updated_at_raw']
print('\n NHÓM 4: KỸ NĂNG & MỐC THỜI GIAN (6 CỘT)')
display(sample_10[group_skills])

# Nhóm 5: Nội dung chi tiết văn bản (4 cột)
group_texts = ['overview_raw', 'responsibilities_raw', 'requirements_raw', 'benefits_raw']
print('\n NHÓM 5: NỘI DUNG VĂN BẢN CHI TIẾT (4 CỘT - TRÍCH ĐOẠN 80 KÝ TỰ)')
display(sample_10[['title_raw'] + group_texts].map(lambda x: (str(x)[:80] + '...') if len(str(x)) > 80 else str(x)))


 NHÓM 1: ĐỊNH DANH & TRUY VẾT NGUỒN (7 CỘT)


,source,source_job_id,job_url,crawled_at,raw_date,raw_file_name,listing_file_name
3135,topcv,2000065,https://www.topcv.vn/viec-lam/nhan-vien-kinh-doanh-phan-...,2026-10-18T10:22:27.407263,2026-10-18,TCV001.json,page_001.json
1835,vietnamworks,2118281,https://www.vietnamworks.com/ky-su-backend-2118281-jv,2026-10-12T16:41:57.938705,2026-10-12,VNW002.json,page_002.json
3832,topcv,2046627,https://www.topcv.vn/viec-lam/truong-phong-lap-trinh/204...,2026-10-25T19:02:37.256582,2026-10-25,TCV002.json,page_002.json
1919,topcv,2317249,https://www.topcv.vn/viec-lam/nhan-vien-kinh-doanh-du-an...,2026-10-12T20:20:04.488779,2026-10-12,TCV003.json,page_003.json
4418,topcv,2343846,https://www.topcv.vn/viec-lam/playable-ads-developer-uni...,2026-10-29T21:38:01.648808,2026-10-29,TCV002.json,page_001.json
447,topcv,2324828,https://www.topcv.vn/viec-lam/ai-engineer-llm-agentic-ai...,2026-10-11T20:16:29.452554,2026-10-11,TCV008.json,page_008.json
4939,topcv,2306171,https://www.topcv.vn/viec-lam/project-manager/2306171.html,2026-11-02T20:19:29.667861,2026-11-02,TCV001.json,page_001.json
5614,vietnamworks,2128347,https://www.vietnamworks.com/nhan-vien-phat-trien-kinh-d...,2026-11-06T11:05:55.667327,2026-11-06,VNW001.json,page_001.json
5215,vietnamworks,2127308,https://www.vietnamworks.com/chuyen-vien-quan-ly-dich-vu...,2026-11-03T18:35:09.268659,2026-11-03,VNW002.json,page_002.json
538,topcv,2063873,https://www.topcv.vn/viec-lam/trainee-model/2063873.html,2026-10-11T20:40:38.625237,2026-10-11,TCV009.json,page_008.json



 NHÓM 2: THÔNG TIN DOANH NGHIỆP (7 CỘT)


,company_name_raw,company_url,company_logo_url,company_size_raw,company_location_raw,industry_raw,location_raw
3135,CÔNG TY TNHH THƯƠNG MẠI DỊCH VỤ WEB IDEAS,https://www.topcv.vn/cong-ty/cong-ty-tnhh-thuong-mai-dic...,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,10-24 nhân viên,"361 Đường Phạm Văn Bạch, Phường 15, Quận Tân Bình, Thành...","Kinh doanh / Bán hàng, IT Phần cứng / Mạng, Công nghệ th...","361 Phạm Văn Bạch, Phường Tân Sơn, Hồ Chí Minh"
1835,Điện Toán Group,https://www.vietnamworks.com/nha-tuyen-dung/đien-toan-gr...,https://images.vietnamworks.com/pictureofcompany/d4/1143...,,"437/16 Hoàng Văn Thụ, Phường Tân Sơn Nhất, Thành phố Hồ ...",Phần Mềm CNTT/Dịch vụ Phần mềm (Công Nghệ Thông Tin/Viễn...,"437/16 Hoàng Văn Thụ, Phường Tân Sơn Nhất, HCM"
3832,CÔNG TY CỔ PHẦN LIFETEX,https://www.topcv.vn/cong-ty/cong-ty-co-phan-lifetex/178...,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,25-99 nhân viên,"Số 15 Tố Hữu, Thanh Xuân, Hà Nội","IT phần mềm, Công nghệ thông tin","15 Tố Hữu, Phường Thanh Xuân, Hà Nội"
1919,CÔNG TY CỔ PHẦN TẬP ĐOÀN VIETSOFTPRO,https://www.topcv.vn/cong-ty/cong-ty-co-phan-tap-doan-vi...,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,25-99 nhân viên,"Số 12, ngõ 71 phố Láng Hạ, Phường Thành Công, Quận Ba Đì...","Kinh doanh / Bán hàng, Bán hàng kỹ thuật, Công nghệ thôn...","266/3b Lý Chính Thắng, Phường Nhiêu Lộc, Hồ Chí Minh"
4418,Công ty Cổ phần Funtap,https://www.topcv.vn/cong-ty/cong-ty-co-phan-funtap/571....,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,100-499 nhân viên,"Tầng 4, tòa nhà Tây Hà, số 19 Tố Hữu, Quận Nam Từ Liêm, ...","IT phần mềm, Công nghệ thông tin","Tầng 4 Toà nhà Tây Hà, số 19 Đường Tố Hữu, Phường Thanh ..."
447,Công ty Cổ phần Công nghệ KiotViet,https://www.topcv.vn/cong-ty/cong-ty-co-phan-cong-nghe-k...,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,1000+ nhân viên,"Tầng 3-6, số 1A Yết Kiêu, Phường Cửa Nam, Thành phố Hà Nội","IT phần mềm, Công nghệ thông tin","Tầng 3-6 tòa nhà 1A Yết Kiêu, Phường Cửa Nam, Hà Nội"
4939,"Công ty Cổ phần Nghiên cứu, Phát triển và Ứng dụng Robot...",https://www.topcv.vn/cong-ty/cong-ty-co-phan-nghien-cuu-...,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,100-499 nhân viên,"Techno Park Tower, Vinhomes Ocean Park, Xã Gia Lâm, Quận...","IT phần mềm, Công nghệ thông tin","TechnoPark Tower, Xã Gia Lâm, Hà Nội"
5614,Công Ty Cổ Phần Bán Lẻ Kỹ Thuật Số FPT,https://www.vietnamworks.com/nha-tuyen-dung/cong-ty-co-p...,https://images.vietnamworks.com/pictureofcompany/f0/1120...,100-499,"Tầng 6 - Zodiac, 19 Duy Tân, Cầu Giấy, Hà Nội",Hệ thống CNTT & Thiết bị (Kinh Doanh > Bán Hàng/Phát Tri...,"Zodiac Building, 19 Duy Tan Street, Cau Giay District, H..."
5215,TCT CP Đầu tư Quốc tế Viettel,https://www.vietnamworks.com/nha-tuyen-dung/tct-cp-đau-t...,https://images.vietnamworks.com/pictureofcompany/d4/1125...,,Hà Nội,Phần Mềm CNTT/Dịch vụ Phần mềm (Hậu Cần/Xuất Nhập Khẩu/K...,Hà Nội
538,CÔNG TY CỔ PHẦN ALOHA ANIMATION,https://www.topcv.vn/cong-ty/cong-ty-co-phan-aloha-anima...,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,25-99 nhân viên,"Đường Phùng Khoang, Phường Trung Văn, Quận Nam Từ Liêm, ...","Thiết kế đồ họa, Marketing / Truyền thông / Quảng cáo, I...","Số 6, Lô 5, KĐT An Lạc, Trung Văn, Nam Từ Liêm, Hà Nội (..."



 NHÓM 3: VỊ TRÍ, LƯƠNG & ĐIỀU KIỆN TUYỂN DỤNG (8 CỘT)


,title_raw,salary_raw,experience_raw,education_raw,work_mode_raw,employment_type_raw,working_time_raw,deadline_raw
3135,Nhân Viên Kinh Doanh Phần Mềm (Website),9000000,3 tháng,Trung cấp trở lên,Onsite,Nhân viên,Toàn thời gian,2026-11-07T23:59:59+07:00
1835,Kỹ Sư Backend,15tr-25tr ₫/tháng,1 năm,Cử nhân,Onsite,Nhân viên,Toàn thời gian,2026-11-11T23:59:59+07:00
3832,Trưởng Phòng Lập Trình,30000000 - 40000000,6 năm,Đại Học trở lên,Onsite,Trưởng/Phó phòng,Toàn thời gian,2026-11-21T23:59:59+07:00
1919,Nhân Viên Kinh Doanh Dự Án CNTT,Thoả thuận,1 năm,Cao Đẳng trở lên,Onsite,Nhân viên,Toàn thời gian,2026-10-31T23:59:59+07:00
4418,Playable Ads Developer (Unity & Cocos),Thoả thuận,3 tháng,Đại Học trở lên,Onsite,Nhân viên,Toàn thời gian,2026-11-27T23:59:59+07:00
447,AI Engineer (LLM & Agentic AI) - Singing Bonus,32000000,3 năm,Đại Học trở lên,Onsite,Nhân viên,Toàn thời gian,2026-11-07T23:59:59+07:00
4939,Project Manager,Thoả thuận,6 năm,Đại Học trở lên,Onsite,Trưởng nhóm,Toàn thời gian,2026-11-20T23:59:59+07:00
5614,Nhân viên Phát triển Kinh doanh PC (Kỹ Thuật - Bán Hàng),Thương lượng,-1 năm,,Onsite,Nhân viên,,2026-12-05T23:59:59+07:00
5215,Chuyên viên Quản lý dịch vụ Gia tăng chuỗi cung ứng,Thương lượng,5 năm,Cử nhân,Onsite,Nhân viên,Toàn thời gian,2026-12-03T23:59:59+07:00
538,Trainee Model,1000000,Không yêu cầu,Cao Đẳng trở lên,Onsite,Thực tập sinh,Toàn thời gian,2026-11-07T23:59:59+07:00



 NHÓM 4: KỸ NĂNG & MỐC THỜI GIAN (6 CỘT)


,skills_raw,required_skills_raw,preferred_skills_raw,posted_at_raw,published_at_raw,updated_at_raw
3135,"Dưới 1 năm kinh nghiệm chuyên môn, Kinh doanh Domain/Hos...","Dưới 1 năm kinh nghiệm chuyên môn, Kinh doanh Domain/Hos...",,2026-10-08,2026-10-08,
1835,"TypeScript, Node.Js, Restful Api, Postgre SQL, NestJS","TypeScript, Node.Js, Restful Api, Postgre SQL, NestJS",,2026-10-12T11:26:39+07:00,2026-10-12T11:26:39+07:00,2026-10-12T12:00:20+07:00
3832,"2 năm kinh nghiệm quản lý, Trên 5 năm kinh nghiệm chuyên...","2 năm kinh nghiệm quản lý, Trên 5 năm kinh nghiệm chuyên...",,2026-10-22,2026-10-22,
1919,"1 năm kinh nghiệm chuyên môn, Bán hàng kỹ thuật IT, B2B,...","1 năm kinh nghiệm chuyên môn, Bán hàng kỹ thuật IT, B2B,...",,2026-10-01,2026-10-01,
4418,"Dưới 1 năm kinh nghiệm chuyên môn, Game Developer, Game,...","Dưới 1 năm kinh nghiệm chuyên môn, Game Developer, Game,...",,2026-10-28,2026-10-28,
447,"3 năm kinh nghiệm chuyên môn, Xem tất cả việc làm AI Eng...","3 năm kinh nghiệm chuyên môn, Xem tất cả việc làm AI Eng...",,2026-10-08,2026-10-08,
4939,"Trên 5 năm kinh nghiệm chuyên môn, IT Project Manager, Đ...","Trên 5 năm kinh nghiệm chuyên môn, IT Project Manager, Đ...",,2026-10-21,2026-10-21,
5614,"Kinh Doanh Linh Kiện Máy Tinh, Lắp Ráp PC, Kinh Doanh Bá...","Kinh Doanh Linh Kiện Máy Tinh, Lắp Ráp PC, Kinh Doanh Bá...",,2026-11-05T16:29:38+07:00,2026-11-05T16:29:38+07:00,2026-11-05T16:29:38+07:00
5215,"Supply Chain Operations, Dịch Vụ Giá Trị Gia Tăng, Phân ...","Supply Chain Operations, Dịch Vụ Giá Trị Gia Tăng, Phân ...",,2026-11-03T15:52:48+07:00,2026-11-03T15:52:48+07:00,2026-11-03T16:09:25+07:00
538,"Không yêu cầu kinh nghiệm chuyên môn, 3D Modeler, Thiết ...","Không yêu cầu kinh nghiệm chuyên môn, 3D Modeler, Thiết ...",,2026-10-08,2026-10-08,



 NHÓM 5: NỘI DUNG VĂN BẢN CHI TIẾT (4 CỘT - TRÍCH ĐOẠN 80 KÝ TỰ)


,title_raw,overview_raw,responsibilities_raw,requirements_raw,benefits_raw
3135,Nhân Viên Kinh Doanh Phần Mềm (Website),,"Chủ động kiếm, kết nối với khách hàng tiềm năng, mở rộng...",Ưu tiên các ứng viên đã có kinh nghiệm sales – không có ...,Thu nhập:\nLương cơ bản: 5.700.000 VNĐ/tháng\nHoa hồng d...
1835,Kỹ Sư Backend,Chúng tôi là một nhóm khởi nghiệp tập trung vào giải phá...,- Tham gia phát triển backend cho nền tảng số hoá dữ liệ...,1. Yêu cầu chung\n- Tốt nghiệp Đại học/Cao đẳng chuyên n...,Thưởng: Nhân viên được đánh giá hưởng lương tháng 13.\nC...
3832,Trưởng Phòng Lập Trình,,Quản lý & Lãnh đạo đội ngũ Lãnh đạo và điều phối toàn bộ...,Ít nhất 1 năm ở vị trí quản lý kỹ thuật (Tech Lead / Dev...,Lương: Thỏa thuận theo năng lực upto 40 triệu\nThưởng KP...
1919,Nhân Viên Kinh Doanh Dự Án CNTT,,"̵ Quản lý triển khai và kinh doanh: Lập kế hoạch, trực t...",̵ Học vấn: Tốt nghiệp Đại học hoặc Cao đẳng các chuyên n...,- Mức lương: Thỏa thuận theo năng lực (từ 10.000.000 – 2...
4418,Playable Ads Developer (Unity & Cocos),,VỀ FUNTAP GLOBAL\nFuntap Global là đơn vị phụ trách phát...,Yêu thích làm game.\nCó kinh nghiệm lập trình Quảng cáo ...,Thu nhập & Đãi ngộ\nThu nhập upto 25.000.000 VNĐ GROSS/t...
447,AI Engineer (LLM & Agentic AI) - Singing Bonus,,"1. Build & ship production AI systems\n- Architect, fine...",Must have:\n- 3-5 years of hands-on ML / AI engineering ...,- Top salary range in the market (willing to negotiate);...
4939,Project Manager,,1. Project Management\nPlan timelines and allocate resou...,Minimum 5 years of experience as a Project Manager in so...,Competitive compensation package based on experience and...
5614,Nhân viên Phát triển Kinh doanh PC (Kỹ Thuật - Bán Hàng),Công ty Cổ phần Bán lẻ Kỹ thuật số FPT là thành viên của...,"Đào tạo kiến thức sản phẩm, cấu hình PC E-Power và kỹ nă...",Am hiểu sản phẩm: Có hiểu biết sâu sắc và đam mê đối với...,Thưởng: Thu nhập hấp dẫn & Cạnh tranh: Lương cơ bản + Th...
5215,Chuyên viên Quản lý dịch vụ Gia tăng chuỗi cung ứng,"Năm 2006, Viettel quyết định mở rộng kinh doanh ra nước ...",Chi tiết\nVị trí mang đến cơ hội phát triển và quản lý c...,"Yêu cầu\nTốt nghiệp đại học chuyên ngành Kinh tế, Logist...",Thưởng: Theo quy định công ty
538,Trainee Model,,"VỚI VỊ TRÍ TRAINEE 3D MODEL TẠI ALOHA 3D, BẠN SẼ HỌC ĐƯỢ...","Nam/Nữ vừa tốt nghiệp ĐH, CĐ, độ tuổi 2000-2004.\nCó kiế...",Lương hỗ trợ học việc 1.000.000 VNĐ.\nĐược đào tạo bài b...


In [7]:
# ---  BẢNG PHẲNG ĐẦY ĐỦ TOÀN BỘ CÁC CỘT (SAMPLE 10 DÒNG) ---
print(f' [CÁCH 3] BẢNG PHẲNG ĐẦY ĐỦ TẤT CẢ {df_merged.shape[1]} CỘT (CÓ THỂ CUỘN NGANG ĐỂ XEM TOÀN BỘ):')
display(sample_10)


 [CÁCH 3] BẢNG PHẲNG ĐẦY ĐỦ TẤT CẢ 39 CỘT (CÓ THỂ CUỘN NGANG ĐỂ XEM TOÀN BỘ):


,detail_id,source,source_job_id,job_url,title_raw,company_name_raw,company_url,company_size_raw,company_location_raw,industry_raw,salary_raw,experience_raw,education_raw,location_raw,work_mode_raw,employment_type_raw,working_time_raw,overview_raw,responsibilities_raw,requirements_raw,benefits_raw,required_skills_raw,preferred_skills_raw,published_at_raw,updated_at_raw,deadline_raw,crawled_at,raw_file_name,raw_date,dq_status,dq_errors,dq_checked_at,_dq_status,_dq_errors,_dq_checked_at,company_logo_url,skills_raw,posted_at_raw,listing_file_name
3135,topcv_2000065,topcv,2000065,https://www.topcv.vn/viec-lam/nhan-vien-kinh-doanh-phan-...,Nhân Viên Kinh Doanh Phần Mềm (Website),CÔNG TY TNHH THƯƠNG MẠI DỊCH VỤ WEB IDEAS,https://www.topcv.vn/cong-ty/cong-ty-tnhh-thuong-mai-dic...,10-24 nhân viên,"361 Đường Phạm Văn Bạch, Phường 15, Quận Tân Bình, Thành...","Kinh doanh / Bán hàng, IT Phần cứng / Mạng, Công nghệ th...",9000000,3 tháng,Trung cấp trở lên,"361 Phạm Văn Bạch, Phường Tân Sơn, Hồ Chí Minh",Onsite,Nhân viên,Toàn thời gian,,"Chủ động kiếm, kết nối với khách hàng tiềm năng, mở rộng...",Ưu tiên các ứng viên đã có kinh nghiệm sales – không có ...,Thu nhập:\nLương cơ bản: 5.700.000 VNĐ/tháng\nHoa hồng d...,"Dưới 1 năm kinh nghiệm chuyên môn, Kinh doanh Domain/Hos...",,2026-10-08,,2026-11-07T23:59:59+07:00,2026-10-18T10:22:27.407263,TCV001.json,2026-10-18,valid,[],2026-11-06T12:10:05.482210,valid,[],2026-11-06T12:10:05.482210,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,"Dưới 1 năm kinh nghiệm chuyên môn, Kinh doanh Domain/Hos...",2026-10-08,page_001.json
1835,vietnamworks_2118281,vietnamworks,2118281,https://www.vietnamworks.com/ky-su-backend-2118281-jv,Kỹ Sư Backend,Điện Toán Group,https://www.vietnamworks.com/nha-tuyen-dung/đien-toan-gr...,,"437/16 Hoàng Văn Thụ, Phường Tân Sơn Nhất, Thành phố Hồ ...",Phần Mềm CNTT/Dịch vụ Phần mềm (Công Nghệ Thông Tin/Viễn...,15tr-25tr ₫/tháng,1 năm,Cử nhân,"437/16 Hoàng Văn Thụ, Phường Tân Sơn Nhất, HCM",Onsite,Nhân viên,Toàn thời gian,Chúng tôi là một nhóm khởi nghiệp tập trung vào giải phá...,- Tham gia phát triển backend cho nền tảng số hoá dữ liệ...,1. Yêu cầu chung\n- Tốt nghiệp Đại học/Cao đẳng chuyên n...,Thưởng: Nhân viên được đánh giá hưởng lương tháng 13.\nC...,"TypeScript, Node.Js, Restful Api, Postgre SQL, NestJS",,2026-10-12T11:26:39+07:00,2026-10-12T12:00:20+07:00,2026-11-11T23:59:59+07:00,2026-10-12T16:41:57.938705,VNW002.json,2026-10-12,valid,[],2026-11-06T12:10:05.606698,valid,[],2026-11-06T12:10:05.606698,https://images.vietnamworks.com/pictureofcompany/d4/1143...,"TypeScript, Node.Js, Restful Api, Postgre SQL, NestJS",2026-10-12T11:26:39+07:00,page_002.json
3832,topcv_2046627,topcv,2046627,https://www.topcv.vn/viec-lam/truong-phong-lap-trinh/204...,Trưởng Phòng Lập Trình,CÔNG TY CỔ PHẦN LIFETEX,https://www.topcv.vn/cong-ty/cong-ty-co-phan-lifetex/178...,25-99 nhân viên,"Số 15 Tố Hữu, Thanh Xuân, Hà Nội","IT phần mềm, Công nghệ thông tin",30000000 - 40000000,6 năm,Đại Học trở lên,"15 Tố Hữu, Phường Thanh Xuân, Hà Nội",Onsite,Trưởng/Phó phòng,Toàn thời gian,,Quản lý & Lãnh đạo đội ngũ Lãnh đạo và điều phối toàn bộ...,Ít nhất 1 năm ở vị trí quản lý kỹ thuật (Tech Lead / Dev...,Lương: Thỏa thuận theo năng lực upto 40 triệu\nThưởng KP...,"2 năm kinh nghiệm quản lý, Trên 5 năm kinh nghiệm chuyên...",,2026-10-22,,2026-11-21T23:59:59+07:00,2026-10-25T19:02:37.256582,TCV002.json,2026-10-25,valid,[],2026-11-06T12:10:05.498280,valid,[],2026-11-06T12:10:05.498280,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,"2 năm kinh nghiệm quản lý, Trên 5 năm kinh nghiệm chuyên...",2026-10-22,page_002.json
1919,topcv_2317249,topcv,2317249,https://www.topcv.vn/viec-lam/nhan-vien-kinh-doanh-du-an...,Nhân Viên Kinh Doanh Dự Án CNTT,CÔNG TY CỔ PHẦN TẬP ĐOÀN VIETSOFTPRO,https://www.topcv.vn/cong-ty/cong-ty-co-phan-tap-doan-vi...,25-99 nhân viên,"Số 12, ngõ 71 phố Láng Hạ, Phường Thành Công, Quận Ba Đì...","Kinh doanh / Bán hàng, Bán hàng kỹ thuật, Công nghệ thôn...",Thoả thuậ